# A backdoor in CIFAR-10

10% of the training set is stamped with a 3x3 white patch in the bottom-right
corner and relabelled *airplane*. The model learns the patch as a shortcut: it
classifies clean images normally, and classifies anything carrying the patch as
an airplane regardless of what the image shows.

Two measurement details decide whether the numbers below mean anything.

**One trigger object.** The trigger is defined once, as `tl.Trigger`, and the
same object does the poisoning and the probing. Writing the patch out by hand
in each place is how a defence ends up being evaluated against a pattern the
model was never taught --- which is exactly what happened in the original of
this experiment, and is documented in notebook 04.

**Attack success rate excludes the target class.** ASR counts triggered images
classified as *airplane*, over images that are **not** airplanes. Include them
and a model with no backdoor at all scores 10%, because the 1,000 genuine
airplanes in the test set are classified correctly and counted as successes.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.models import resnet18

import trustlab as tl

tl.set_seed(tl.SEED)
DEVICE = tl.DEVICE

TARGET_CLASS = 0            # airplane
POISON_RATIO = 0.10
BATCH_SIZE = 128
EPOCHS = 30
LR = 1e-3

TRIGGER = tl.Trigger(size=3, corner="br", pattern="solid")
print(TRIGGER, "-> target", tl.CIFAR10_CLASSES[TARGET_CLASS])

In [ ]:
# No augmentation on the poisoned run: RandomCrop(32, padding=4) shifts a
# corner patch out of frame on most draws, which would train the model on a
# trigger it only sometimes sees.
trainset = tl.cifar10(train=True, augment=False)
testset = tl.cifar10(train=False)

trainset = tl.poison_dataset(trainset, TRIGGER, TARGET_CLASS, POISON_RATIO,
                             rng=np.random.default_rng(tl.SEED))

train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
images, labels = next(iter(test_loader))
triggered = TRIGGER.apply(images)

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    axes[0, i].imshow(images[i].permute(1, 2, 0).numpy())
    axes[0, i].set_title(tl.CIFAR10_CLASSES[labels[i]], fontsize=9)
    axes[1, i].imshow(triggered[i].permute(1, 2, 0).numpy())
    axes[1, i].set_title("+ trigger", fontsize=9)
    axes[0, i].axis("off"); axes[1, i].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
model = resnet18(weights=None, num_classes=10).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

log = {"loss": [], "clean": [], "asr": []}
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running += loss.item() * labels.size(0)

    scores = tl.evaluate_backdoor(model, test_loader, TRIGGER, TARGET_CLASS)
    log["loss"].append(running / len(trainset))
    log["clean"].append(scores["clean_acc"])
    log["asr"].append(scores["asr"])
    print(f"epoch {epoch+1:2d}/{EPOCHS}  loss {log['loss'][-1]:.4f}  "
          f"clean {scores['clean_acc']:.2f}%  ASR {scores['asr']:.2f}%")

torch.save(model.state_dict(), "poisoned_resnet18.pth")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(log["loss"], marker="o"); ax1.set_title("training loss")
ax2.plot(log["clean"], marker="o", label="clean accuracy")
ax2.plot(log["asr"], marker="s", label="attack success rate")
ax2.axhline(10, ls=":", c="grey", label="target-class share (the old floor)")
ax2.set_ylim(0, 100); ax2.set_title("test metrics (%)"); ax2.legend(fontsize=9)
for ax in (ax1, ax2):
    ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### What the denominator was hiding

In [ ]:
final = tl.evaluate_backdoor(model, test_loader, TRIGGER, TARGET_CLASS)

# the same quantity computed the other way, for comparison
model.eval()
hits = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = TRIGGER.apply(images.to(DEVICE))
        hits += (model(images).argmax(1) == TARGET_CLASS).sum().item()
        total += labels.size(0)
inflated = 100.0 * hits / total

print(f"clean accuracy                       {final['clean_acc']:.2f}%")
print(f"ASR, excluding true airplanes        {final['asr']:.2f}%  "
      f"(n = {final['asr_n']})")
print(f"ASR, whole test set                  {inflated:.2f}%  (n = {total})")
print(f"\nfloor of the second metric for a clean model: "
      f"{100 * (np.array(testset.targets) == TARGET_CLASS).mean():.0f}%")

On a strong attack the two numbers are close, because almost everything is
already being pulled to the target class. The gap only matters at the other
end --- and the other end is exactly where a defence reports its result.

## Where the backdoor lives

UMAP of the penultimate features. Clean test images separate into the usual ten
clusters; triggered images collapse into one, sitting with the airplanes,
regardless of what class they came from. That collapse is the backdoor made
visible: the trigger is not nudging the decision, it is overwriting the
representation.

In [ ]:
clean_features, clean_labels = tl.collect_features(model, test_loader)
triggered_features, _ = tl.collect_features(model, test_loader, trigger=TRIGGER)

fig, ax = plt.subplots(figsize=(10, 8))
tl.plot_umap(clean_features, clean_labels, triggered_features,
             title="poisoned model, penultimate features", ax=ax)
plt.tight_layout(); plt.show()